In [36]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 0 : CONFIGURATION                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

from pyspark.sql import functions as F
from pyspark.sql.types import *

storage_account = "energybigdatastorage"
container_raw = "raw"
container_processed = "processed"

path_raw = f"abfss://{container_raw}@{storage_account}.dfs.core.windows.net/energy_data_extracted/archive (3).zip/weather_daily_darksky.csv"
path_processed = f"abfss://{container_processed}@{storage_account}.dfs.core.windows.net/weather_daily/"

print(f"Source: {path_raw}")
print(f"Destination: {path_processed}")

In [37]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 : INGESTION                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path_raw)

print(f"Nombre de lignes: {df.count()}")
print(f"Nombre de colonnes: {len(df.columns)}")
print(f"\n=== SCHEMA ===")
df.printSchema()
print(f"\n=== APERCU ===")
df.show(3, truncate=False)

In [38]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 : PROFILING AVANT NETTOYAGE                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("=== VALEURS NULLES ===")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

print(f"\n=== DOUBLONS ===")
print(f"Nombre de doublons: {df.count() - df.dropDuplicates().count()}")

print(f"\n=== INSPECTION DATE ===")
df.select("time").show(5, truncate=False)

print(f"\n=== STATISTIQUES COLONNES NUMERIQUES ===")
numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))]
print(f"Colonnes numeriques: {numeric_cols}")

for c in numeric_cols:
    stats = df.select(
        F.min(c).alias("min"),
        F.max(c).alias("max"),
        F.avg(c).alias("avg"),
        F.stddev(c).alias("stddev"),
        F.count(F.when(F.col(c).isNull(), c)).alias("nulls")
    ).collect()[0]
    print(f"{c:25s} | min={stats.min:10.2f} | max={stats.max:10.2f} | avg={stats.avg:10.2f} | std={stats.stddev:10.2f} | nulls={stats.nulls}")

print(f"\n=== COLONNES STRING (TOP 10) ===")
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType) and f.name != "time"]
for c in string_cols:
    print(f"\n--- {c} ---")
    df.groupBy(c).count().orderBy(F.desc("count")).show(10, truncate=False)

In [39]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 : NETTOYAGE                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# 1. Convertir date
df_clean = df.withColumn("date", F.to_date(F.col("time"), "yyyy-MM-dd"))
# Si format standard: F.to_date(F.col("time"))

null_date = df_clean.filter(F.col("date").isNull()).count()
df_clean = df_clean.filter(F.col("date").isNotNull())
print(f"Lignes sans date supprimees: {null_date}")

# 2. Features temporelles
df_clean = df_clean \
    .withColumn("year", F.year(F.col("date"))) \
    .withColumn("month", F.month(F.col("date"))) \
    .withColumn("dayofweek", F.dayofweek(F.col("date"))) \
    .withColumn("is_weekend", F.when(F.col("dayofweek").isin([1, 7]), 1).otherwise(0)) \
    .withColumn("quarter", F.quarter(F.col("date"))) \
    .withColumn("day_of_year", F.dayofyear(F.col("date")))

# 3. Colonnes meteo
daily_cols = ["temperatureMax", "temperatureMin", "temperatureHigh", "temperatureLow",
              "apparentTemperatureMax", "apparentTemperatureMin",
              "humidity", "windSpeed", "windBearing", "visibility",
              "cloudCover", "pressure", "precipIntensity", "precipProbability",
              "dewPoint", "uvIndex", "ozone"]
existing_cols = [c for c in daily_cols if c in df_clean.columns]
print(f"Colonnes detectees: {existing_cols}")

# 4. Conversion + NaN -> NULL
for c in existing_cols:
    df_clean = df_clean.withColumn(c, F.col(c).cast("double"))
    df_clean = df_clean.withColumn(c, F.when(F.isnan(F.col(c)), F.lit(None)).otherwise(F.col(c)))

# 5. Consistance Tmax >= Tmin
temp_max_col = "temperatureMax" if "temperatureMax" in existing_cols else "temperatureHigh"
temp_min_col = "temperatureMin" if "temperatureMin" in existing_cols else "temperatureLow"

if temp_max_col in existing_cols and temp_min_col in existing_cols:
    inconsistency = df_clean.filter(F.col(temp_max_col) < F.col(temp_min_col)).count()
    print(f"Inconsistances Tmax < Tmin detectees: {inconsistency}")
    
    df_clean = df_clean.withColumn("temp_consistency",
        F.when(F.col(temp_max_col) < F.col(temp_min_col), 0).otherwise(1))
    
    # Corriger en echangeant
    df_clean = df_clean.withColumn(f"{temp_max_col}_fixed",
        F.when(F.col("temp_consistency") == 0, F.col(temp_min_col)).otherwise(F.col(temp_max_col))
    ).withColumn(f"{temp_min_col}_fixed",
        F.when(F.col("temp_consistency") == 0, F.col(temp_max_col)).otherwise(F.col(temp_min_col))
    )
    
    df_clean = df_clean.drop(temp_max_col, temp_min_col) \
        .withColumnRenamed(f"{temp_max_col}_fixed", temp_max_col) \
        .withColumnRenamed(f"{temp_min_col}_fixed", temp_min_col)
    
    print(f"Inconsistances corrigees par echange")

# 6. Humidite [0, 1]
if "humidity" in existing_cols:
    df_clean = df_clean.withColumn("humidity",
        F.when(F.col("humidity") > 1, F.col("humidity") / 100)
        .when((F.col("humidity") < 0) | (F.col("humidity") > 1), F.lit(None))
        .otherwise(F.col("humidity"))
    )

# 7. Pression [900, 1100] hPa
if "pressure" in existing_cols:
    absurd_press = df_clean.filter((F.col("pressure") < 900) | (F.col("pressure") > 1100)).count()
    df_clean = df_clean.withColumn("pressure",
        F.when((F.col("pressure") < 900) | (F.col("pressure") > 1100), F.lit(None))
        .otherwise(F.col("pressure"))
    )
    print(f"Pressions absurdes corrigees: {absurd_press}")

# 8. Supprimer doublons
dupes = df_clean.count() - df_clean.dropDuplicates(["date"]).count()
df_clean = df_clean.dropDuplicates(["date"])
print(f"Doublons date supprimes: {dupes}")

# 9. Trier
df_clean = df_clean.orderBy("date")

# 10. Metadata
df_clean = df_clean.withColumn("processed_date", F.current_date())

print(f"\nNombre de lignes nettoyees: {df_clean.count()}")
df_clean.select("date", temp_max_col, temp_min_col, "humidity", "pressure").show(5)

In [40]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 4 : STATISTIQUES APRES NETTOYAGE                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("=== VALEURS NULLES ===")
df_clean.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in existing_cols]).show()

print("\n=== DISTRIBUTION TEMPORRELLE ===")
df_clean.select(
    F.min("date").alias("date_min"),
    F.max("date").alias("date_max"),
    F.countDistinct("date").alias("nb_jours")
).show()

print("\n=== COMPARAISON WEEKEND vs SEMAINE ===")
df_clean.groupBy("is_weekend").agg(
    F.round(F.avg(temp_max_col), 2).alias(f"avg_{temp_max_col}"),
    F.round(F.avg(temp_min_col), 2).alias(f"avg_{temp_min_col}"),
    F.count("*").alias("nb_jours")
).show()

In [41]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 5 : SAUVEGARDE DELTA                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year", "month") \
    .save(path_processed)

print("Sauvegarde terminee dans processed/weather_daily/")

In [42]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 : VERIFICATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

df_verify = spark.read.format("delta").load(path_processed)
print(f"Verification: {df_verify.count()} lignes")
print(f"\n=== SCHEMA ===")
df_verify.printSchema()
print(f"\n=== APERCU ===")
df_verify.select("date", temp_max_col, temp_min_col, "humidity", "pressure", "year", "month").show(5)

print(f"\n=== TESTS RAPIDES ===")
print(f"date NULL: {df_verify.filter(F.col('date').isNull()).count()}")
print(f"Tmax < Tmin: {df_verify.filter(F.col(temp_max_col) < F.col(temp_min_col)).count()}")
print(f"Humidite hors [0,1]: {df_verify.filter((F.col('humidity') < 0) | (F.col('humidity') > 1)).count()}")
print(f"Pression hors [900,1100]: {df_verify.filter((F.col('pressure') < 900) | (F.col('pressure') > 1100)).count()}")
print(f"Doublons date: {df_verify.count() - df_verify.dropDuplicates(['date']).count()}")